# SYNTHETIC SPEECH

In [32]:
root = "data/torgo/processed/audio/"
speeches = [
    "F/F01/Session1/wav_arrayMic/0001.wav",
    "F/F03/Session2/wav_arrayMic/s2_0422.wav",
    "F/F04/Session2/wav_arrayMic/s2_0204.wav"
]
speeches = [root + speech for speech in speeches]

In [36]:
from IPython.display import Audio
Audio(speeches[1])

In [4]:
from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

/Users/lkieu/PycharmProjects/Audio-Health-Benchmark/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:08<00:00,  1.03it/s]


In [37]:

wav = tts.tts("I hope my data can help manage the pandemic.",
                speaker_wav=speeches,
                language="en")

In [38]:
Audio(wav, rate=22050)

# Generate 200 Synthetic Recordings from C9S_T1
Select 20 people (10 positive, 10 negative; 5M/5F each). Generate 200 new recordings (100 from positive, 100 from negative). Each uses 3 speaker wavs chosen without replacement, no duplicate triplet.

In [1]:
import pandas as pd
import random
import os

random.seed(42)

df = pd.read_csv('data/c9s_t1/processed/c9s_t1.csv')
audio_root = 'data/c9s_t1/processed/audio/'

def get_unique_participants(df, label, sex):
    return df[(df['label'] == label) & (df['Sex'] == sex)].drop_duplicates('Participant_ID')

def pick_diverse_symptoms(sub, n=5):
    """Pick n participants with diverse symptom patterns."""
    groups = list(sub.groupby('Symptoms'))
    random.shuffle(groups)
    selected = []
    seen = set()
    for sym, grp in groups:
        if len(selected) >= n:
            break
        if sym not in seen:
            selected.append(grp.sample(1, random_state=42))
            seen.add(sym)
    return pd.concat(selected)

# Positive: pick 5M + 5F with diverse symptoms
pos_males = pick_diverse_symptoms(get_unique_participants(df, 1, 'Male'), 37)
pos_females = pick_diverse_symptoms(get_unique_participants(df, 1, 'Female'), 38)

# Negative: pick 5M + 5F randomly
neg_males = get_unique_participants(df, 0, 'Male').sample(13, random_state=42)
neg_females = get_unique_participants(df, 0, 'Female').sample(12, random_state=42)

positive_pids = pos_males['Participant_ID'].tolist() + pos_females['Participant_ID'].tolist()
negative_pids = neg_males['Participant_ID'].tolist() + neg_females['Participant_ID'].tolist()

def get_audio_paths(pids):
    """Get all existing audio file paths for a list of participant IDs."""
    paths = []
    for pid in pids:
        pid_paths = df[df['Participant_ID'] == pid]['path'].tolist()
        existing = [audio_root + p for p in pid_paths if os.path.exists(audio_root + p)]
        paths.extend(existing)
    return paths

positive_paths = get_audio_paths(positive_pids)
negative_paths = get_audio_paths(negative_pids)

print(f"Positive participants ({len(positive_pids)}):")
for _, r in pd.concat([pos_males, pos_females]).iterrows():
    print(f"  {r['Participant_ID']} | {r['Sex']} | {r['Age']} | Symptoms: {r['Symptoms']}")

print(f"\nNegative participants ({len(negative_pids)}):")
for _, r in pd.concat([neg_males, neg_females]).iterrows():
    print(f"  {r['Participant_ID']} | {r['Sex']} | {r['Age']}")

print(f"\nPositive audio files: {len(positive_paths)}")
print(f"Negative audio files: {len(negative_paths)}")
print(f"\nPositive paths: {positive_paths}")
print(f"Negative paths: {negative_paths}")

Positive participants (75):
  bdInlKShvWDZ | Male | 60-69 | Symptoms: drycough,shortbreath,tightness,headache,muscleache,smelltasteloss,dizziness
  PDE6HEWV92qm | Male | 20-29 | Symptoms: drycough,wetcough,shortbreath,tightness,dizziness
  1M96W0fJhqAQ | Male | 40-49 | Symptoms: drycough,shortbreath,muscleache
  8hAq4vSEa10c | Male | 20-29 | Symptoms: drycough,sorethroat,dizziness
  yC5sOXLOR8 | Male | 30-39 | Symptoms: drycough,wetcough,headache
  HVUzMD7ueXZh | Male | 40-49 | Symptoms: drycough,tightness,headache,smelltasteloss
  NmXA4FCk1Onn | Male | 20-29 | Symptoms: wetcough,shortbreath,dizziness
  SAEbtGBRT16e | Male | 16-19 | Symptoms: wetcough,headache,sorethroat,dizziness
  T0755AMWpwoU | Male | 30-39 | Symptoms: drycough,runnyblockednose
  PPbpvAz6Ye | Male | 40-49 | Symptoms: wetcough,tightness,dizziness,sorethroat,runny
  R1FqTju2a0Ag | Male | 20-29 | Symptoms: wetcough,headache,muscleache,sorethroat,runnyblockednose
  325LdkFh7XeU | Male | 30-39 | Symptoms: drycough,headac

In [2]:
from itertools import combinations

# Pre-generate all unique combo (no duplicate choice)
pos_combos = [list(c) for c in combinations(positive_paths, 2)]
neg_combos = [list(c) for c in combinations(negative_paths, 2)]

random.shuffle(pos_combos)
random.shuffle(neg_combos)

print(f"Available positive combos: {len(pos_combos)} (need 750)")
print(f"Available negative combos: {len(neg_combos)} (need 250)")

assert len(pos_combos) >= 750, f"Not enough positive combos: {len(pos_combos)}"
assert len(neg_combos) >= 250, f"Not enough negative combos: {len(neg_combos)}"

pos_combos = pos_combos[:750]
neg_combos = neg_combos[:250]

Available positive triplets: 3486 (need 750)
Available negative triplets: 561 (need 250)


In [5]:
import soundfile as sf
import numpy as np
import tempfile

output_dir = 'data/c9s_t1/synthetic1000'
os.makedirs(f'{output_dir}/positive', exist_ok=True)
os.makedirs(f'{output_dir}/negative', exist_ok=True)

text = "I hope my data can help manage the pandemic."
MAX_TOTAL_SEC = 6

def cut_middle(wav_paths, max_total_sec=MAX_TOTAL_SEC):
    """Load each wav, cut per_audio seconds from the middle, return temp file paths."""
    per_audio_sec = max_total_sec / len(wav_paths)
    tmp_paths = []
    for p in wav_paths:
        data, sr = sf.read(p)
        per_audio_samples = int(per_audio_sec * sr)
        if len(data) > per_audio_samples:
            start = (len(data) - per_audio_samples) // 2
            data = data[start : start + per_audio_samples]
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        sf.write(tmp.name, data, sr)
        tmp_paths.append(tmp.name)
    return tmp_paths

# Generate 750 positive recordings
for i, combo in enumerate(pos_combos):
    out_path = f'{output_dir}/positive/syn_pos_{i:04d}.wav'
    print(f"[{i+1}/750] Positive: {[os.path.basename(p) for p in combo]}")
    cut_paths = cut_middle(combo)
    wav = tts.tts(text, speaker_wav=cut_paths, language="en")
    sf.write(out_path, np.array(wav), 22050)
    for p in cut_paths:
        os.unlink(p)

# Generate 250 negative recordings
for i, combo in enumerate(neg_combos):
    out_path = f'{output_dir}/negative/syn_neg_{i:04d}.wav'
    print(f"[{i+1}/250] Negative: {[os.path.basename(p) for p in combo]}")
    cut_paths = cut_middle(combo)
    wav = tts.tts(text, speaker_wav=cut_paths, language="en")
    sf.write(out_path, np.array(wav), 22050)
    for p in cut_paths:
        os.unlink(p)

print(f"\nDone! Generated 1000 recordings in {output_dir}/")

[1/750] Positive: ['voice_DclLQnsFnk_1603377532121.wav', 'voice_OwWhCSjrTe_1613604523976.wav']


/Users/lkieu/PycharmProjects/Audio-Health-Benchmark/.venv/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


[2/750] Positive: ['audio_file_read.wav', 'audio_file_read.wav']
[3/750] Positive: ['audio_file_read.wav', 'audio_file_read.wav']
[4/750] Positive: ['voice_NLB9ZuU8n2_1594105848500.wav', 'audio_file_read.wav']
[5/750] Positive: ['audio_file_read.wav', 'audio_file_read.wav']
[6/750] Positive: ['voice_1LsH8soFmm_1604935639231.wav', 'audio_file_read.wav']
[7/750] Positive: ['audio_file_read.wav', 'audio_file_read.wav']
[8/750] Positive: ['voice_PrBDhwbkjb_1608540706742.wav', 'audio_file_read.wav']
[9/750] Positive: ['voice_DclLQnsFnk_1603377532121.wav', 'voice_SyqbkzhEfb_1606682285759.wav']
[10/750] Positive: ['audio_file_read.wav', 'voice_SyqbkzhEfb_1606682285759.wav']
[11/750] Positive: ['audio_file_read.wav', 'audio_file_read.wav']
[12/750] Positive: ['voice_z7r48yOrHR_1607188876974.wav', 'audio_file_read.wav']
[13/750] Positive: ['audio_file_read.wav', 'audio_file_read.wav']
[14/750] Positive: ['voice_1LsH8soFmm_1604935639231.wav', 'voice_DclLQnsFnk_1603377532121.wav']
[15/750] Positi

# Encode synthetic recordings with XTTS
Extract `gpt_cond_latent` and `speaker_embedding` from each synthetic wav. Save to HDF5.

In [1]:
import torch
import h5py
import glob

# Patch torch.load for TTS 0.22 compatibility
_orig_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

from model.xtts import XTTS

encoder = XTTS(
    model_name="tts_models/multilingual/multi-dataset/xtts_v2",
    freeze_encoder=True,
    sample_rate=22050,
)

output_dir = 'data/c9s_t1/synthetic1000'
h5_path = f'{output_dir}/xtts_embeddings.hdf5'

pos_wavs = sorted(glob.glob(f'{output_dir}/positive/syn_pos_*.wav'))
neg_wavs = sorted(glob.glob(f'{output_dir}/negative/syn_neg_*.wav'))

print(f"Found {len(pos_wavs)} positive, {len(neg_wavs)} negative wavs")

with h5py.File(h5_path, 'w') as f:
    for i, wav_path in enumerate(pos_wavs):
        key = f"pos_{i:03d}"
        print(f"[{i+1}/{len(pos_wavs)}] Encoding {key}")
        gpt_cond, spk_emb = encoder._extract_from_path(wav_path)
        grp = f.create_group(key)
        grp.create_dataset("gpt_cond_latent", data=gpt_cond.squeeze(0).cpu().numpy())
        grp.create_dataset("speaker_embedding", data=spk_emb.squeeze().cpu().numpy())

    for i, wav_path in enumerate(neg_wavs):
        key = f"neg_{i:03d}"
        print(f"[{i+1}/{len(neg_wavs)}] Encoding {key}")
        gpt_cond, spk_emb = encoder._extract_from_path(wav_path)
        grp = f.create_group(key)
        grp.create_dataset("gpt_cond_latent", data=gpt_cond.squeeze(0).cpu().numpy())
        grp.create_dataset("speaker_embedding", data=spk_emb.squeeze().cpu().numpy())

print(f"\nSaved embeddings to {h5_path}")

# Verify
with h5py.File(h5_path, 'r') as f:
    keys = list(f.keys())
    print(f"Total keys: {len(keys)}")
    sample = f[keys[0]]
    print(f"Sample '{keys[0]}': gpt_cond_latent={sample['gpt_cond_latent'].shape}, speaker_embedding={sample['speaker_embedding'].shape}")

/Users/lkieu/PycharmProjects/Audio-Health-Benchmark/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 30393.51it/s]


Found 750 positive, 250 negative wavs
[1/750] Encoding pos_000
[2/750] Encoding pos_001


/Users/lkieu/PycharmProjects/Audio-Health-Benchmark/.venv/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


[3/750] Encoding pos_002
[4/750] Encoding pos_003
[5/750] Encoding pos_004
[6/750] Encoding pos_005
[7/750] Encoding pos_006
[8/750] Encoding pos_007
[9/750] Encoding pos_008
[10/750] Encoding pos_009
[11/750] Encoding pos_010
[12/750] Encoding pos_011
[13/750] Encoding pos_012
[14/750] Encoding pos_013
[15/750] Encoding pos_014
[16/750] Encoding pos_015
[17/750] Encoding pos_016
[18/750] Encoding pos_017
[19/750] Encoding pos_018
[20/750] Encoding pos_019
[21/750] Encoding pos_020
[22/750] Encoding pos_021
[23/750] Encoding pos_022
[24/750] Encoding pos_023
[25/750] Encoding pos_024
[26/750] Encoding pos_025
[27/750] Encoding pos_026
[28/750] Encoding pos_027
[29/750] Encoding pos_028
[30/750] Encoding pos_029
[31/750] Encoding pos_030
[32/750] Encoding pos_031
[33/750] Encoding pos_032
[34/750] Encoding pos_033
[35/750] Encoding pos_034
[36/750] Encoding pos_035
[37/750] Encoding pos_036
[38/750] Encoding pos_037
[39/750] Encoding pos_038
[40/750] Encoding pos_039
[41/750] Encoding p